In [1]:
from __future__ import annotations
%pip install -r requirements.txt
from shutil import which


if which("nvidia-smi"):
    print("Detected NVIDIA driver: installing CUDA-enabled PyTorch")
    %pip install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
else:
    print("No NVIDIA driver detected: installing CPU-only PyTorch")
    %pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.
Detected NVIDIA driver: installing CUDA-enabled PyTorch



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (2449.3 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-win_amd64.whl (6.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (4.1 MB)

   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   -----------------------------------


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
import time
import pandas as pd
from datasets import load_dataset
from elasticsearch import Elasticsearch, helpers
import requests
from ipywidgets import Text, Button, HBox, VBox, Output, Layout, HTML
from IPython.display import display
import random
from torch.utils.data import DataLoader, Dataset
from collections import Counter
from tqdm.auto import tqdm

from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import math
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

import warnings, logging, contextlib, os, sys

@contextlib.contextmanager
def _suppress_model_noise():
    # Hide the specific deprecation line and similar noisy messages
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"`torch_dtype` is deprecated! Use `dtype` instead!",
            category=DeprecationWarning,
        )
        # Temporarily silence stderr while models initialize (some libs print directly there)
        with open(os.devnull, "w") as devnull, contextlib.redirect_stderr(devnull):
            yield



In [52]:
ds = load_dataset("calmgoose/amazon-product-data-2020")
streaming_ds = load_dataset("calmgoose/amazon-product-data-2020", split="train", streaming=True)
streaming_ds,next(iter(streaming_ds))




(IterableDataset({
     features: ['Uniq Id', 'Product Name', 'Category', 'Upc Ean Code', 'Selling Price', 'Model Number', 'About Product', 'Product Specification', 'Technical Details', 'Shipping Weight', 'Product Dimensions', 'Image', 'Variants', 'Product Url', 'Is Amazon Seller'],
     num_shards: 1
 }),
 {'Uniq Id': '4c69b61db1fc16e7013b43fc926e502d',
  'Product Name': 'DB Longboards CoreFlex Crossbow 41" Bamboo Fiberglass Longboard Complete',
  'Category': 'Sports & Outdoors | Outdoor Recreation | Skates, Skateboards & Scooters | Skateboarding | Standard Skateboards & Longboards | Longboards',
  'Upc Ean Code': None,
  'Selling Price': '$237.68',
  'Model Number': None,
  'About Product': "Make sure this fits by entering your model number. | RESPONSIVE FLEX: The Crossbow features a bamboo core encased in triaxial fiberglass and HD plastic for a responsive flex pattern that’s second to none. Pumping & carving have never been so satisfying! Flex 2 is recommended for people 120 to 170

In [31]:
#Start Elasticsearch server
ES_URL = "http://127.0.0.1:9200"
!docker run -d --name es-dev -p 9200:9200 -e "discovery.type=single-node" -e "xpack.security.enabled=false" docker.elastic.co/elasticsearch/elasticsearch:8.14.0
for i in range(60):
    try:
        r = requests.get(ES_URL, timeout=2)
        if r.ok:
            print("Elasticsearch is up:", r.json().get("cluster_name"), r.json().get("version", {}).get("number"))
            break
    except Exception as e:
        time.sleep(2)
else:
    raise RuntimeError("Elasticsearch did not start within the expected time")


Elasticsearch is up: docker-cluster 8.14.0


docker: Error response from daemon: Conflict. The container name "/es-dev" is already in use by container "15d63b7e35e506c2273abdafa79d629d377a999d9dfe89d7756341afbabc81cc". You have to remove (or rename) that container to be able to reuse that name.

Run 'docker run --help' for more information


In [32]:
# Connect explicitly to the Docker container endpoint started in the previous cell
# Use options suitable for local, insecure HTTP and add robust retry behavior
es = Elasticsearch(
    ES_URL,
    request_timeout=30,
    retry_on_timeout=True,
    verify_certs=False  # local HTTP container (no TLS)
)

# Use a direct info() call (more reliable than ping) and allow more time/backoff for the container to be ready
last_err = None
for i in range(10):  # up to ~120s with 2s sleeps
    try:
        info = es.info()
        print("Connected to Elasticsearch container:", info.get("cluster_name"), info.get("version", {}).get("number"))
        break
    except Exception as e:
        last_err = e
        time.sleep(2)
else:
    raise RuntimeError(
        f"Could not connect to Elasticsearch container at {ES_URL}. Ensure Docker container 'es-dev' is running and port 9200 is published.\n"
        f"Last error: {type(last_err).__name__}: {last_err}"
    )


Connected to Elasticsearch container: docker-cluster 8.14.0


In [47]:
INDEX = "amazon_products_2020"

# Ensure mappings for uniq_key and collapse_key exist for field collapsing
try:
    es.indices.create(index=INDEX, ignore=400)
    es.indices.put_mapping(index=INDEX, body={"properties": {"uniq_key": {"type": "keyword"}, "collapse_key": {"type": "keyword"}}})
except Exception as e:
    print("Warning: could not ensure mapping for uniq_key:", e)

# Stream to avoid downloading everything at once
streaming_ds = load_dataset("calmgoose/amazon-product-data-2020", split="train", streaming=True)

def to_action(ex):
    doc = dict(ex)
    # Build a stable ID from preferred unique fields; overwrite on re-index
    candidates = [
        doc.get("Uniq Id"),
        doc.get("UniqId"),
        doc.get("uniq_id"),
        doc.get("asin"),
        doc.get("Product Url"), doc.get("Product URL"), doc.get("product_url"),
        doc.get("Model Number"), doc.get("model_number"),
        doc.get("Upc Ean Code"), doc.get("UPC"), doc.get("EAN"),
        doc.get("Product Name"), doc.get("title"), doc.get("name"),
    ]
    key = None
    for v in candidates:
        if isinstance(v, str) and v.strip():
            key = v.strip()
            break
    if not key:
        # Deterministic fallback: SHA1 of selected fields
        import hashlib
        sig_fields = [
            "Uniq Id", "UniqId", "uniq_id", "asin",
            "Product Url", "Product URL", "product_url",
            "Model Number", "model_number",
            "Upc Ean Code", "UPC", "EAN",
            "Product Name", "title", "name",
            "About Product", "Product Specification", "Technical Details", "Category", "Selling Price"
        ]
        parts = []
        for k in sig_fields:
            v = doc.get(k)
            if v is None:
                continue
            if isinstance(v, str):
                vv = v.strip()
                if not vv:
                    continue
                parts.append(f"{k}={vv}")
            else:
                parts.append(f"{k}={v}")
        sig = "|".join(parts)
        key = hashlib.sha1(sig.encode("utf-8", errors="ignore")).hexdigest()
    # also store uniq_key in _source so we can use ES collapsing
    doc["uniq_key"] = key
    # compute a looser collapse key from normalized title; fallback to uniq_key if title missing
    try:
        title = (
            doc.get("Product Name")
            or doc.get("product_name")
            or doc.get("title")
            or doc.get("name")
        )
        if isinstance(title, str) and title.strip():
            import re
            t = title.lower()
            # replace non-alphanumeric with space, collapse multiple spaces
            t = re.sub(r"[^a-z0-9]+", " ", t)
            t = re.sub(r"\s+", " ", t).strip()
            collapse_key = t if t else key
        else:
            collapse_key = key
    except Exception:
        collapse_key = key
    doc["collapse_key"] = collapse_key
    return {
        "_index": INDEX,
        "_id": key,
        "_source": doc
    }

batch = []
BATCH_SIZE = 2000
count = 0

for ex in streaming_ds:
    batch.append(to_action(ex))
    if len(batch) >= BATCH_SIZE:
        helpers.bulk(es, batch)
        count += len(batch)
        batch.clear()

# Flush remainder
if batch:
    helpers.bulk(es, batch)
    count += len(batch)

print(f"Indexed {count} documents")

C:\Users\linic\AppData\Local\Temp\ipykernel_27704\3196900698.py:5: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es.indices.create(index=INDEX, ignore=400)


Indexed 10002 documents


In [53]:
# Important fields for candidate_text construction
DEFAULT_FIELDS: Tuple[str, ...] = (
    "Product Name",
    "About Product",
    "Product Specification",
    "Technical Details",
)


def _first_non_empty(d: Dict[str, Any], keys: Sequence[str]) -> Optional[str]:
    for k in keys:
        v = d.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return None


def build_candidate_text(
    source: Dict[str, Any],
    fields: Sequence[str] = DEFAULT_FIELDS,
    max_chars_per_field: int = 300,
    max_total_chars: int = 1200,
    sep: str = " | ",
) -> str:
    """
    Build a concise candidate text from ES _source using important fields.

    - Picks a title from common keys.
    - Appends selected fields, each truncated to max_chars_per_field.
    - Ensures total length stays reasonably small for cross-encoders.
    """
    parts: List[str] = []

    # Prefer a title-like field
    title = _first_non_empty(source, ["Product Name", "product_name", "title", "name"])
    if title:
        parts.append(title.strip())

    # Add important fields
    total_len = sum(len(p) for p in parts)
    for f in fields:
        v = source.get(f)
        if not isinstance(v, str):
            continue
        s = " ".join(v.split())  # squash whitespace
        if not s:
            continue
        if max_chars_per_field:
            s = s[: max_chars_per_field].rstrip()
        parts.append(f"{f}: {s}")
        total_len += len(parts[-1])
        if max_total_chars and total_len >= max_total_chars:
            break

    # Fallback: include some other possibly relevant text fields if nothing else
    if len(parts) == 0:
        for k, v in source.items():
            if isinstance(v, str) and v.strip():
                parts.append(" ".join(v.split())[: max_chars_per_field])
                break

    text = sep.join(parts)
    if max_total_chars and len(text) > max_total_chars:
        text = text[: max_total_chars].rstrip()
    return text


class JinaReranker:
    """Reranker using jinaai/jina-reranker-v2-base-multilingual.

    Scores (query, doc) pairs with a relevance score. Higher is more relevant.
    """

    def __init__(
        self,
        model_name: str = "jinaai/jina-reranker-v2-base-multilingual",
        device: Optional[str] = None,
        batch_size: int = 16,
        max_length: int = 512,
        use_fp16: bool = True,
    ) -> None:
        self.model_name = model_name
        self.batch_size = batch_size
        self.max_length = max_length

        if device is None:
            if torch.cuda.is_available():
                device = "cuda"
            else:
                device = "cpu"
        self.device = torch.device(device)

        logging.getLogger("transformers").setLevel(logging.ERROR)

        # Suppress deprecation + noisy stderr during model init only
        #imports model
        with _suppress_model_noise():
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            self.model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True)
        self.model.to(self.device)
        self.model.eval()

        # Determine positive label index for classification heads if present
        self.pos_label_idx: Optional[int] = None
        id2label = getattr(self.model.config, "id2label", None)
        if isinstance(id2label, dict) and len(id2label) >= 2:
            # Try to find a label that looks like the positive class
            for idx_str, label in id2label.items():
                try:
                    idx = int(idx_str)
                except Exception:
                    # In some configs keys are ints already
                    idx = int(idx_str) if isinstance(idx_str, int) else None
                if idx is None:
                    continue
                if str(label).lower() in {"relevant", "entailment", "pos", "positive"}:
                    self.pos_label_idx = idx
                    break
            if self.pos_label_idx is None:
                # Fall back to the last index
                self.pos_label_idx = max(int(i) for i in id2label.keys())

        self.use_fp16 = use_fp16 and (self.device.type == "cuda")

    @torch.inference_mode()
    def score(self, query: str, docs: Sequence[str]) -> List[float]:
        scores: List[float] = []
        if not docs:
            return scores

        for i in range(0, len(docs), self.batch_size):
            batch_docs = docs[i : i + self.batch_size]
            enc = self.tokenizer(
                [query] * len(batch_docs),
                list(batch_docs),
                truncation=True,
                padding=True,
                max_length=self.max_length,
                return_tensors="pt",
            )
            enc = {k: v.to(self.device) for k, v in enc.items()}

            if self.use_fp16:
                with torch.autocast(device_type=self.device.type, dtype=torch.float16):
                    logits = self.model(**enc).logits
            else:
                logits = self.model(**enc).logits

            if logits.shape[-1] == 1:
                batch_scores = logits.squeeze(-1).detach().float().tolist()
            else:
                # Assume classification; use softmax prob of positive label
                probs = torch.softmax(logits, dim=-1)
                pos_idx = self.pos_label_idx if self.pos_label_idx is not None else logits.shape[-1] - 1
                batch_scores = probs[:, pos_idx].detach().float().tolist()
            scores.extend(batch_scores)
        return scores

    def rank_hits(
        self,
        query: str,
        hits: Sequence[Dict[str, Any]],
        *,
        fields: Sequence[str] = DEFAULT_FIELDS,
        top_n: Optional[int] = None,
        blend_alpha: Optional[float] = None,
        max_chars_per_field: int = 300,
        max_total_chars: int = 1200,
    ) -> List[Dict[str, Any]]:
        """
        Rerank ES hits (list of hits as in resp["hits"]["hits"]). Returns a new sorted list of hits.

        - If blend_alpha is provided in [0,1], blend normalized ES _score with reranker score.
        - Otherwise, sort by reranker score alone.
        """
        if not hits:
            return []

        # Build candidate texts
        cand_texts: List[str] = []
        for h in hits:
            src = h.get("_source", {}) or {}
            cand_texts.append(
                build_candidate_text(
                    src,
                    fields=fields,
                    max_chars_per_field=max_chars_per_field,
                    max_total_chars=max_total_chars,
                )
            )

        rr_scores = self.score(query, cand_texts)

        # Attach scores
        enriched: List[Dict[str, Any]] = []
        for h, rr in zip(hits, rr_scores):
            h2 = dict(h)  # shallow copy
            h2["_rerank_score"] = float(rr)
            enriched.append(h2)

        # Blending with ES _score if requested
        if blend_alpha is not None:
            a = float(max(0.0, min(1.0, blend_alpha)))
            es_scores = [float(h.get("_score", 0.0)) for h in enriched]
            es_norm = _min_max_normalize(es_scores)
            rr_norm = _min_max_normalize([h["_rerank_score"] for h in enriched])
            for i, h in enumerate(enriched):
                h["_final_score"] = a * rr_norm[i] + (1.0 - a) * es_norm[i]
            key = "_final_score"
        else:
            key = "_rerank_score"

        enriched.sort(key=lambda x: x.get(key, -math.inf), reverse=True)
        if top_n is not None:
            enriched = enriched[:top_n]
        return enriched


def _min_max_normalize(values: Sequence[float]) -> List[float]:
    if not values:
        return []
    vmin = min(values)
    vmax = max(values)
    if vmax <= vmin:
        return [0.0 for _ in values]
    return [(v - vmin) / (vmax - vmin) for v in values]


def retrieve_and_rerank(
    es_client: Any,
    index: str,
    query_text: str,
    search_fields: Sequence[str],
    *,
    top_k: int = 100,
    top_n: int = 20,
    highlight_fields: Optional[Sequence[str]] = None,
    blend_alpha: Optional[float] = None,
    model: Optional[JinaReranker] = None,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Executes ES retrieval, then reranks.

    Returns (reranked_hits, raw_hits).
    """
    hl = None
    if highlight_fields:
        hl = {"fields": {f: {} for f in highlight_fields}}

    resp = es_client.search(
        index=index,
        query={"multi_match": {"query": query_text, "fields": list(search_fields)}},
        highlight=hl,
        collapse={"field": "collapse_key"},
        size=top_k,
    )
    hits = resp.get("hits", {}).get("hits", [])

    reranker = model or JinaReranker()
    ranked = reranker.rank_hits(
        query_text,
        hits,
        fields=DEFAULT_FIELDS,
        top_n=top_n,
        blend_alpha=blend_alpha,
    )
    return ranked, hits


In [49]:
search_fields = [
    'Uniq Id',
    'Product Name',
    'Category',
    'Upc Ean Code',
    'Selling Price',
    'Model Number',
    'About Product',
    'Product Specification',
    'Technical Details',
    'Shipping Weight',
    'Product Dimensions',
    'Image',
    'Variants',
    'Product Url',
    'Is Amazon Seller'
]

css = HTML("""
<style>
.custom-search .widget-text input {
  font-size: 16px;
  padding: 10px 14px;
}
.custom-search .widget-button {
  font-size: 16px;
  padding: 8px 14px;
}
/* Bigger label for the Text description ("Query:") */
.custom-search .widget-label {
  font-size: 18px;
}
</style>
""")

q = Text(
    value="wireless noise cancelling headphones",
    placeholder="Type your search…",
    description="Query:",
    disabled=False,
    layout=Layout(width="600px", height="40px"),
    style={"description_width": "90px"}  # make room for the bigger label
)
q.continuous_update = False

btn = Button(
    description="Search",
    button_style="primary",  # 'primary', 'success', 'info', 'warning', 'danger' or ''
    layout=Layout(width="600px", height="40px")  # match Text width
)

out = Output()


def run_search(query_text: str):
    with out:
        out.clear_output()
        if not query_text.strip():
            print("Please enter a query.")
            return
        # Execute initial retrieval from Elasticsearch (top_k=50)
        resp = es.search(
            index=INDEX,
            query={"multi_match": {"query": query_text, "fields": search_fields}},
            highlight={
                "fields": {
                    "Product Name": {},
                    "About Product": {},
                    "Product Specification": {},
                    "Technical Details": {},
                    "Category": {}
                }
            },
            collapse={"field": "collapse_key"},
            size=50
        )
        hits = resp.get("hits", {}).get("hits", [])
        # Save for reranking in the next cell
        globals()["LAST_QUERY"] = query_text
        globals()["LAST_ES_HITS"] = hits

        # Show initial ES-ranked results
        for hit in hits: #[:5]:
            src = hit.get("_source", {})
            title = src.get("Product Name") or src.get("product_name") or src.get("title") or "(no title)"
            print(f"{hit.get('_score', 0.0):.4f}  {title}")
            if "highlight" in hit:
                for field, frags in hit["highlight"].items():
                    print("  ", field, "=>", " ... ".join(frags))
        print("\nTip: Run the next cell to rerank these results with the Jina reranker.")


def on_text_change(change):
    if change.get("name") == "value":
        run_search(q.value)


def on_click(b):
    run_search(q.value)


q.observe(on_text_change, names="value")
btn.on_click(on_click)

# Center both controls and stack them vertically
controls = VBox([q, btn], layout=Layout(align_items="center", width="100%", gap="10px"))

# Inject CSS and apply a class wrapper for scoping
container = VBox([css, controls, out])
container.add_class("custom-search")

display(container)


In [50]:
# Rerank the results from the previous cell using the Jina multilingual reranker
# - Expects LAST_QUERY and LAST_ES_HITS to be set by the search cell above.
# - Prints results ordered by cross-encoder relevance score.
_prev_query = globals().get("LAST_QUERY", None)
_prev_hits = globals().get("LAST_ES_HITS", None)

if _prev_query is None or _prev_hits is None:
    print("No previous search results found. Please run the search cell first.")
elif not _prev_hits:
    print("No hits to rerank. Try a different query in the search cell.")
else:
    print(f"Reranking {_prev_hits and len(_prev_hits) or 0} hits for query: {_prev_query!r}")
    try:
        reranker = JinaReranker()  # jinaai/jina-reranker-v2-base-multilingual
        print(f"Using reranker device: {getattr(reranker, 'device', '(unknown)')}")
        ranked_hits = reranker.rank_hits(
            _prev_query,
            _prev_hits,
            fields=DEFAULT_FIELDS,
            top_n=1,        # keep all; set an int to truncate
            blend_alpha=None,  # set e.g. 0.3 to blend with ES _score
        )
        for hit in ranked_hits:
            src = hit.get("_source", {})
            title = src.get("Product Name") or src.get("product_name") or src.get("title") or "(no title)"
            score_to_show = hit.get("_rerank_score", hit.get("_score", 0.0))
            print(f"{score_to_show:.4f}  {title}")
            if "highlight" in hit:
                for field, frags in hit["highlight"].items():
                    print("  ", field, "=>", " ... ".join(frags))
    except Exception as e:
        print("Warning: Reranking failed:", e)

Reranking 50 hits for query: 'wireless noise cancelling headphones'
Using reranker device: cuda
-2.2227  Trolls Poppy Kid Friendly Headphones with Built in Volume Limiting Feature for Kid Friendly Safe Listening
   About Product => . | Parental Control: These <em>headphones</em> come equipped with an adjustable volume limiter to provide a safe
   Category => Electronics | <em>Headphones</em> | Over-Ear <em>Headphones</em>
   Product Name => Trolls Poppy Kid Friendly <em>Headphones</em> with Built in Volume Limiting Feature for Kid Friendly Safe Listening
   Technical Details => | Color:Poppy show up to 2 reviews by default The new Trolls Poppy kid friendly <em>headphones</em> are here!! ... These high quality <em>headphones</em> are equipped with a volume limiting switch to ensure safe sound levels ... Check out our innovative <em>headphones</em>, ear buds, speakers, microphones, boom boxes, karaoke recording studios


In [58]:
# Basic training loop for fine-tuning a simple text classifier on product categories

# Load training split (non-streaming)
train_ds = load_dataset("calmgoose/amazon-product-data-2020", split="train")
train_from_scratch=False
# Build input text using fields similar to retrieval/rerank pipeline
search_fields = [
    'Uniq Id',
    'Product Name',
    'Category',
    'Upc Ean Code',
    'Selling Price',
    'Model Number',
    'About Product',
    'Product Specification',
    'Technical Details',
    'Shipping Weight',
    'Product Dimensions',
    'Image',
    'Variants',
    'Product Url',
    'Is Amazon Seller'
]


def _build_text(ex):
    try:
        return build_candidate_text(ex, fields=search_fields, max_chars_per_field=200, max_total_chars=512)
    except Exception:
        # Fallback: concatenate available string fields
        parts = []
        for k, v in ex.items():
            if isinstance(v, str) and v.strip():
                parts.append(v.strip())
        return (" ".join(parts))[:512] if parts else ""


# Collect labeled examples for training
label_col = "Category"
examples = []
for ex in train_ds:
    text = _build_text(ex)
    if not text:
        continue
    lbl = ex.get(label_col)
    if isinstance(lbl, list):
        lbl = lbl[0] if lbl else None
    if lbl is None:
        continue
    lbl = str(lbl).strip()
    if not lbl:
        continue
    examples.append((text, lbl))

if not examples:
    print("No labeled examples found; skipping training.")
else:
    texts, raw_labels = zip(*examples)
    labels_list = sorted(list({l for l in raw_labels}))
    label2id = {l: i for i, l in enumerate(labels_list)}
    id2label = {i: l for l, i in label2id.items()}
    labels = [label2id[l] for l in raw_labels]

    if True:
        # Train/val split
        idxs = list(range(len(texts)))
        random.shuffle(idxs)
        split = int(0.9 * len(idxs))
        train_idx, val_idx = idxs[:split], idxs[split:]


        def _subset(idxs):
            return [texts[i] for i in idxs], [labels[i] for i in idxs]


        train_texts, train_labels = _subset(train_idx)
        val_texts, val_labels = _subset(val_idx)

        if train_from_scratch:
            model_name = "jinaai/jina-reranker-v2-base-multilingual"
        else:
            model_name = "models\\fine_tuned_reranker"
        tokenizer = AutoTokenizer.from_pretrained(model_name)


        class ClsDS(Dataset):
            def __init__(self, texts, labels, tokenizer):
                self.texts = list(texts)
                self.labels = torch.tensor(labels, dtype=torch.long)
                self.tokenizer = tokenizer

            def __len__(self):
                return len(self.labels)

            def __getitem__(self, i):
                enc = self.tokenizer(
                    self.texts[i], padding="max_length", truncation=True, max_length=256, return_tensors="pt"
                )
                item = {k: v.squeeze(0) for k, v in enc.items()}
                item["labels"] = self.labels[i]
                return item


        train_dataset = ClsDS(train_texts, train_labels, tokenizer)
        val_dataset = ClsDS(val_texts, val_labels, tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=16)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("Using device:", device)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=len(labels_list), id2label=id2label, label2id=label2id
        ).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

        # Basic training loop
        epochs = 5
        for epoch in range(epochs):
            model.train()
            pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")
            total_loss = 0.0
            for batch in pbar:
                batch = {k: v.to(device) for k, v in batch.items()}
                optimizer.zero_grad()
                out = model(**batch)
                loss = out.loss
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                if pbar.n:
                    pbar.set_postfix(loss=f"{(total_loss / pbar.n):.4f}")
            print(f"Train loss: {total_loss / max(1, len(train_loader)):.4f}")

            # Validation
            model.eval()
            correct, total, val_loss = 0, 0, 0.0
            with torch.no_grad():
                for batch in val_loader:
                    batch = {k: v.to(device) for k, v in batch.items()}
                    out = model(**batch)
                    val_loss += out.loss.item()
                    preds = out.logits.argmax(dim=-1)
                    correct += (preds == batch["labels"]).sum().item()
                    total += batch["labels"].size(0)
            acc = correct / total if total else 0.0
            print(f"Val loss: {val_loss / max(1, len(val_loader)):.4f} | Val acc: {acc:.3f}")

        print("Training complete.")
        import os
        save_dir = "models\\fine_tuned_reranker"
        os.makedirs(save_dir, exist_ok=True)
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"Saved fine-tuned model to {save_dir}")


Using device: cuda


Epoch 1/5:   0%|          | 0/13 [00:00<?, ?it/s]

Train loss: 2.0793
Val loss: 2.0122 | Val acc: 0.083


Epoch 2/5:   0%|          | 0/13 [00:00<?, ?it/s]

Train loss: 1.7462
Val loss: 1.4128 | Val acc: 0.750


...

Train loss: 0.9572
Val loss: 0.5892 | Val acc: 1.000


Epoch 4/5:   0%|          | 0/13 [00:00<?, ?it/s]

Train loss: 0.4320
Val loss: 0.2522 | Val acc: 1.000


Epoch 5/5:   0%|          | 0/13 [00:00<?, ?it/s]

Train loss: 0.2011
Val loss: 0.1178 | Val acc: 1.000
Training complete.
Saved fine-tuned model to models\fine_tuned_reranker


In [ ]:
# Fine-tuned model reranker
from os import path
save_dir = "models\\fine_tuned_reranker"
if not path.isdir(save_dir):
    print("Fine-tuned model directory not found. Please run the training cell first.")
else:
    try:
        reranker_ft = JinaReranker(model_name=save_dir)
        print(f"Fine-tuned reranker ready. Device: {getattr(reranker_ft, 'device', '(unknown)')}")
        # If there are previous ES results, demonstrate reranking with the fine-tuned model
        _prev_query = globals().get("LAST_QUERY", None)
        _prev_hits = globals().get("LAST_ES_HITS", None)
        if _prev_query and _prev_hits:
            ranked_hits = reranker_ft.rank_hits(
                _prev_query,
                _prev_hits,
                fields=DEFAULT_FIELDS,
                top_n=1,        # keep all; set an int to truncate
                blend_alpha=None,
            )
            for hit in ranked_hits:
                src = hit.get("_source", {})
                title = src.get("Product Name") or src.get("product_name") or src.get("title") or "(no title)"
                score_to_show = hit.get("_rerank_score", hit.get("_score", 0.0))
                print(f"{score_to_show:.4f}  {title}")
        else:
            print("Tip: Run the search cell above, then rerun this cell to rerank with the fine-tuned model.")
    except Exception as e:
        print("Warning: Failed to load fine-tuned reranker:", e)